In [1]:
from datetime import datetime, date
import pandas as pd
import itertools
from finrl.agents.elegantrl.models import DRLAgent
from enum import StrEnum, auto
from finrl.meta.data_processor import DataProcessor
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from stable_baselines3.common.logger import configure
from finrl.meta.env_stock_trading.env_forex_price_trailing import ForexPriceTrailingEnv
import torch as th
from elegantrl.train.config import Config
from elegantrl.train.run    import train_agent
import torch.nn as nns
from elegantrl.agents import AgentDQN, AgentDoubleDQN
from copy import deepcopy
%matplotlib inline

In [2]:
class Currency(StrEnum):
    USD = auto()
    EUR = auto()
    JPY = auto()
    GBP = auto()
    AUD = auto()
    CAD = auto()
    CHF = auto()
    NZD = auto()
    CNY = auto()

In [3]:
start_date: date = date(2010, 1, 1)
end_date: date = date(2020, 1, 1)

majors = [
    "EURUSD=X","USDJPY=X","GBPUSD=X",
    "AUDUSD=X","USDCAD=X","USDCHF=X","NZDUSD=X"
]

# 2) Top non-USD crosses
crosses = [
    "EURGBP=X","EURJPY=X","GBPJPY=X","AUDJPY=X",
    "CADJPY=X","EURAUD=X","EURCAD=X","EURCHF=X",
    "GBPCHF=X","AUDCAD=X","NZDJPY=X","NZDCAD=X"
]

# 3) Key CNY pairs
cny_pairs = [
    "USDCNY=X","EURCNY=X","JPY CNY=X".replace(" ",""),  # -> "JPYCNY=X"
    "GBPCNY=X","AUDCNY=X","CADCNY=X","CHFCNY=X","NZDCNY=X"
]

# 4) Stitch together, then take the first 20 unique
all_tickers = majors + crosses + cny_pairs
# remove any duplicates and slice to 20
seen = set()
forex_ticks: tuple[str, ...] = tuple(
    t for t in all_tickers
    if not (t in seen or seen.add(t))
)
len(forex_ticks)

27

In [4]:
yfd = YahooDownloader(start_date=str(start_date), end_date=str(end_date), ticker_list=forex_ticks)
df = yfd.fetch_data()
df

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (70305, 8)


Price,date,close,high,low,open,volume,tic,day
0,2010-01-01,0.945120,0.945720,0.944000,0.945510,0,AUDCAD=X,4
1,2010-01-01,6.009700,6.037000,6.003100,6.036400,0,AUDCNY=X,4
2,2010-01-01,83.449997,83.602997,83.202003,83.463997,0,AUDJPY=X,4
3,2010-01-01,0.898473,0.898473,0.897827,0.898311,0,AUDUSD=X,4
4,2010-01-01,6.397100,6.397300,6.386700,6.386700,0,CADCNY=X,4
...,...,...,...,...,...,...,...,...
70300,2019-12-31,0.673478,0.675621,0.672088,0.673559,0,NZDUSD=X,1
70301,2019-12-31,1.306060,1.306080,1.295310,1.305800,0,USDCAD=X,1
70302,2019-12-31,0.968630,0.969680,0.964500,0.968640,0,USDCHF=X,1
70303,2019-12-31,6.985700,6.985900,6.957500,6.985700,0,USDCNY=X,1


In [5]:
def add_fx_features_for_tick(g: pd.DataFrame) -> pd.DataFrame:
    g["close_prev"] = g.close.shift(1)
    g["high_prev"] = g.high.shift(1)
    g["low_prev"] = g.low.shift(1)

    g["x1"] = (g.close - g.close_prev) / g.close_prev
    g["x2"] = (g.high - g.high_prev) / g.high_prev
    g["x3"] = (g.low - g.low_prev) / g.low_prev
    g["x4"] = (g.high - g.close) / g.close
    g["x5"] = (g.close - g.low) / g.close
    return g

def add_fx_features(df: pd.DataFrame, tic_col: str = "tic") -> pd.DataFrame:
    df_with_features = (
        df
        .groupby(tic_col, group_keys=False)
        .apply(add_fx_features_for_tick)
        .drop(columns=["close_prev", "high_prev", "low_prev"])
        .fillna(0)
    )
    return df_with_features

In [6]:
df_features = add_fx_features(df)
df_features

/var/folders/23/n0prkghs6v94ssbsv_9xq74c0000gn/T/ipykernel_62449/2277383529.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)


Price,date,close,high,low,open,volume,tic,day,x1,x2,x3,x4,x5
0,2010-01-01,0.945120,0.945720,0.944000,0.945510,0,AUDCAD=X,4,0.000000,0.000000,0.000000,0.000635,0.001185
1,2010-01-01,6.009700,6.037000,6.003100,6.036400,0,AUDCNY=X,4,0.000000,0.000000,0.000000,0.004543,0.001098
2,2010-01-01,83.449997,83.602997,83.202003,83.463997,0,AUDJPY=X,4,0.000000,0.000000,0.000000,0.001833,0.002972
3,2010-01-01,0.898473,0.898473,0.897827,0.898311,0,AUDUSD=X,4,0.000000,0.000000,0.000000,0.000000,0.000718
4,2010-01-01,6.397100,6.397300,6.386700,6.386700,0,CADCNY=X,4,0.000000,0.000000,0.000000,0.000031,0.001626
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70300,2019-12-31,0.673478,0.675621,0.672088,0.673559,0,NZDUSD=X,1,0.004229,0.002703,0.002204,0.003182,0.002063
70301,2019-12-31,1.306060,1.306080,1.295310,1.305800,0,USDCAD=X,1,-0.000979,-0.001743,-0.007653,0.000015,0.008231
70302,2019-12-31,0.968630,0.969680,0.964500,0.968640,0,USDCHF=X,1,-0.005217,-0.004435,-0.003101,0.001084,0.004264
70303,2019-12-31,6.985700,6.985900,6.957500,6.985700,0,USDCNY=X,1,-0.001301,-0.001272,-0.002766,0.000029,0.004037


## Custom LSTM Q-value estimator

In [7]:
import torch as th
import torch.nn.functional as F
from torch import nn

# your paper-style LSTM block:
class LSTM_QNet(nn.Module):
    def __init__(
        self,
        window:      int = 16,
        feature_dim: int = 5,
        lstm_hidden: int = 32,
        fc_hidden:   int = 64,
        action_dim:  int = 3,
    ):
        super().__init__()
        self.window      = window
        self.feature_dim = feature_dim
        self.action_dim  = action_dim

        # LSTM over the last `window` frames of 5-dim features
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
        )
        # one small FC after the LSTM
        self.fc_after_lstm = nn.Linear(lstm_hidden, lstm_hidden)
        # two more FCs after concatenating the prev-pos one-hot
        self.fc1 = nn.Linear(lstm_hidden + action_dim, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden,       fc_hidden)
        # final head: Q for each action
        self.q_head = nn.Linear(fc_hidden, action_dim)

    def forward(
        self,
        state: th.Tensor
    ) -> th.Tensor:
        q_value = self.get_q_value(state)
        return q_value.argmax(dim=1)

    def get_q_value(self, state: th.Tensor) -> th.Tensor:
        B = state.size(0)
        # split into the history window and the prev-pos scalar
        hist = state[:, : self.window * self.feature_dim]
        prev = state[:, -1].long()  # δ ∈ {-1,0,1}
        # reshape history into (B, window, feature_dim)
        seq = hist.view(B, self.window, self.feature_dim)

        # run LSTM
        lstm_out, _ = self.lstm(seq)      # -> (B, window, lstm_hidden)
        h_T     = lstm_out[:, -1, :]      # last step hidden: (B, lstm_hidden)
        h       = F.relu(self.fc_after_lstm(h_T))

        # one-hot encode prev position: map {-1,0,1} -> {0,1,2} then one_hot
        oh = F.one_hot(prev + 1, num_classes=self.action_dim).float()  # (B, action_dim)

        # concat and finish
        z  = th.cat([h, oh], dim=1)       # (B, lstm_hidden+action_dim)
        z1 = F.relu(self.fc1(z))
        z2 = F.relu(self.fc2(z1))
        q  = self.q_head(z2)              # (B, action_dim)
        return q

    def get_action(
        self,
        state: th.Tensor,
        explore_rate: float
    ) -> th.Tensor:
        """
        ε-greedy over Q-values.
        state: (B, state_dim)
        explore_rate: float in [0,1)
        returns: (B,1) tensor of discrete actions
        """
        if explore_rate < th.rand(1, device=state.device):
            # choose best Q
            act = self.get_q_value(state).argmax(dim=1, keepdim=True)
        else:
            # random
            B = state.size(0)
            act = th.randint(self.action_dim, (B,1), device=state.device)
        return act


In [8]:
# ---------- 1.2  Wrap this in an ElegantRL agent -------------------
class AgentDDQN_LSTM(AgentDQN):
    def __init__(
        self, 
        net_dim, 
        state_dim, 
        action_dim,
        gpu_id=0,
        args=None,
    ):
        super().__init__(net_dim, state_dim, action_dim, gpu_id=gpu_id, args=args)
        self.act = LSTM_QNet(
            window=16, 
            feature_dim=5,
            lstm_hidden=128, 
            action_dim=action_dim
        ).to(self.device)
        self.act_target = deepcopy(self.act)
        self.cri = self.act
        self.cri_target = self.act_target
        self.criterion = nn.SmoothL1Loss()

In [11]:
window: int = 16

cfg = Config(
    agent_class = AgentDDQN_LSTM,      
    env_class   = ForexPriceTrailingEnv,
    env_args    = dict(
        env_name  ='ForexTrail_v2',
        num_envs  = 1,
        max_step  = 1000,
        state_dim = 5 * window + 1,
        action_dim= 3,
        if_discrete=True,
        full_df   = df_features,
        window    = window,
        margin    = 0.02,
        step_frac = 0.1,
        pick_new_pair_every = 700,
        gpu_id = 0
    )
)
cfg.gamma           = 0.999
cfg.horizon_len     = 1024
cfg.batch_size      = 512
cfg.repeat_times    = 4
cfg.eval_per_step   = 3_000
cfg.eval_times      = 2
cfg.break_step      = 2_000_000        # stop after 1 M env steps
cfg.net_dims        = []               # unused (we override net)

train_agent(cfg, if_single_process=True)

| train_agent_single_process() with GPU_ID 0
| Arguments Remove cwd: ./ForexTrail_v2_DDQN_LSTM_0
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Critic network. Or call it loss function of critic network.
| `objA`: Objective of Actor network. It is the average Q value of the critic network.
################################################################################
ID     Step    Time |    avgR   stdR   avgS  stdS |    expR   objC   objA   etc.
0  1.02e+03       2 |-6309985.0034084.3   1000     0 |-6004.416037.01   0.03   0.25 [ 44  40 414]
0  4.10e+03       8 |-6285492.00134262.6   1000     0 |-5989.406049.82   0.03   0.25 [ 

KeyboardInterrupt: 

In [ ]:
# from elegantrl.train.config import Config
# from elegantrl.train.run import train_agent  # the usual entrypoint
# from elegantrl.agents import AgentDiscreteA2C
# import os
# %matplotlib inline

# os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
# window = 16

# # 1) Build the Config
# cfg = Config(
#     agent_class=AgentDiscreteA2C,                
#     env_class=ForexPriceTrailingEnv,
#     env_args={
#         'env_name':     'ForexTrail',    # just for naming/saving
#         'num_envs':     1,               # single-env rollout\\
#         'env_num':      1,
#         'max_step':     2000,            # EP_LEN in your code
#         'state_dim':    5 * window + 1,    # obs-dimension
#         'action_dim':   3,               # Discrete(3)
#         'if_discrete':  True,
#         # any other init kwargs your env needs:
#         "episode_len": 1000,
#         "alpha_trail": 0.7,
#         "alpha_pnl": 1.0,
#         "alpha_fee": 0.8,
#         'full_df':      df_features,
#         'window':       16,
#         'margin':       0.02,
#         'step_frac':    0.1,
#         "pick_new_pair_every": 700,
#         "live_plot": True
#     }
# )

# # 2) Tweak evaluation settings
# cfg.eval_per_step = 2000     # “gap” between successive evaluations (in total training steps) 
# cfg.eval_times    = 3     # # of episodes to run when evaluating, then average their returns
# cfg.gamma = 0.999
# cfg.break_step = 1e7

# # 3) Kick off training + periodic eval
# train_agent(cfg, if_single_process=True)